In [ ]:
pip install tensorflow numpy pandas matplotlib pillow tqdm nltk scikit-learn

In [1]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image
import pickle
import nltk
import tensorflow as tf

In [2]:
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, Add
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

In [3]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gmano\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
zip_path = 'archive (5).zip'
extract_path = 'flickr_dataset'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print('Dataset extracted successfully!')

Dataset extracted successfully!


In [5]:
BASE_DIR = 'flickr_dataset'
IMAGES_DIR = os.path.join(BASE_DIR, 'Images')
CAPTION_FILE = os.path.join(BASE_DIR, 'captions.txt')

In [6]:
def load_captions(filename):
    captions = {}

    with open(filename, 'r', encoding='utf-8') as file:
        next(file)

        for line in file:
            tokens = line.strip().split(',', 1)
            if len(tokens) < 2:
                continue

            image_id, caption = tokens
            caption = 'startseq ' + caption.lower() + ' endseq'

            if image_id not in captions:
                captions[image_id] = []

            captions[image_id].append(caption)

    return captions

captions_mapping = load_captions(CAPTION_FILE)
print('Total Images:', len(captions_mapping))

Total Images: 8091


In [7]:
base_model = InceptionV3(weights='imagenet')
model = Model(base_model.input, base_model.layers[-2].output)

In [ ]:
def extract_features(directory):
    features = {}

    for img_name in tqdm(os.listdir(directory)):
        img_path = os.path.join(directory, img_name)

        image = load_img(img_path, target_size=(299, 299))
        image = img_to_array(image)
        image = np.expand_dims(image, axis=0)
        image = preprocess_input(image)

        feature = model.predict(image, verbose=0)
        image_id = img_name
        features[image_id] = feature

    return features

features = extract_features(IMAGES_DIR)

with open('features.pkl', 'wb') as f:
    pickle.dump(features, f)

  0%|                                                                                         | 0/8091 [00:00<?, ?it/s]

  0%|                                                                               | 9/8091 [00:17<1:59:19,  1.13it/s]

In [ ]:
all_captions = []
for key in captions_mapping:
    all_captions.extend(captions_mapping[key])

tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)

vocab_size = len(tokenizer.word_index) + 1
max_length = max(len(caption.split()) for caption in all_captions)

print('Vocabulary Size:', vocab_size)
print('Maximum Caption Length:', max_length)

In [ ]:
image_ids = list(captions_mapping.keys())
train_ids, test_ids = train_test_split(image_ids, test_size=0.2, random_state=42)

In [ ]:
def data_generator(image_ids, captions, features, tokenizer, max_length, vocab_size, batch_size):
    X1, X2, y = [], [], []
    n = 0

    while True:
        for image_id in image_ids:
            n += 1
            image_feature = features[image_id][0]

            for caption in captions[image_id]:
                seq = tokenizer.texts_to_sequences([caption])[0]

                for i in range(1, len(seq)):
                    in_seq, out_seq = seq[:i], seq[i]

                    in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
                    out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]

                    X1.append(image_feature)
                    X2.append(in_seq)
                    y.append(out_seq)

            if n == batch_size:
                yield ({'image_input': np.array(X1),
                        'text_input': np.array(X2)},
                       np.array(y))

                X1, X2, y = [], [], []
                n = 0

In [ ]:
inputs1 = Input(shape=(2048,), name='image_input')
fe1 = Dropout(0.4)(inputs1)
fe2 = Dense(256, activation='relu')(fe1)

inputs2 = Input(shape=(max_length,), name='text_input')
se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
se2 = Dropout(0.4)(se1)
se3 = LSTM(256)(se2)

decoder1 = Add()([fe2, se3])
decoder2 = Dense(256, activation='relu')(decoder1)
outputs = Dense(vocab_size, activation='softmax')(decoder2)

caption_model = Model(inputs=[inputs1, inputs2], outputs=outputs)
caption_model.compile(loss='categorical_crossentropy', optimizer='adam')

caption_model.summary()

In [ ]:
epochs = 1
batch_size = 64
steps = len(train_ids) // batch_size

for epoch in range(epochs):
    generator = data_generator(
        train_ids,
        captions_mapping,
        features,
        tokenizer,
        max_length,
        vocab_size,
        batch_size
    )

    caption_model.fit(generator,
                      epochs=1,
                      steps_per_epoch=steps,
                      verbose=1)

    caption_model.save(f'image_caption_model_{epoch+1}.h5')

In [ ]:
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

In [ ]:
def idx_to_word(integer, tokenizer):
    for word, index in tokenizer.word_index.items():
        if index == integer:
            return word
    return None


def predict_caption(model, image_feature, tokenizer, max_length):
    in_text = 'startseq'

    for _ in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length)

        yhat = model.predict([image_feature, sequence], verbose=0)
        yhat = np.argmax(yhat)

        word = idx_to_word(yhat, tokenizer)
        if word is None:
            break

        in_text += ' ' + word

        if word == 'endseq':
            break

    return in_text.replace('startseq', '').replace('endseq', '').strip()

In [ ]:
def extract_single_image_feature(image_path):
    image = load_img(image_path, target_size=(299, 299))
    image = img_to_array(image)
    image = np.expand_dims(image, axis=0)
    image = preprocess_input(image)

    feature = model.predict(image, verbose=0)
    return feature

image_path = 'test.jpg'
feature = extract_single_image_feature(image_path)

caption = predict_caption(
    caption_model,
    feature,
    tokenizer,
    max_length
)

print('Generated Caption:', caption)

img = Image.open(image_path)
plt.imshow(img)
plt.axis('off')
plt.show()